executive_dashboard

In [1]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT/"fact_quote.csv")

claim = pd.read_csv(FACT/"fact_claim.csv")

uw = pd.read_csv(FACT/"fact_underwriting.csv")

journey = pd.read_csv(FACT/"fact_customer_journey.csv")

ai = pd.read_csv(FACT/"fact_ai_interaction.csv")

channel = pd.read_csv(DIM/"dim_channel.csv")

customer = pd.read_csv(DIM/"dim_customer.csv")

policy = pd.read_csv(DIM/"dim_policy.csv")

vehicle = pd.read_csv(DIM/"dim_vehicle.csv")

# ==========================================================
# Quote KPIs
# ==========================================================

total_quotes = len(quote)

converted = quote["accepted_offer"].sum()

conversion_rate = round(
    converted/total_quotes*100,
    2
)

avg_premium = round(
    quote["quoted_premium"].mean(),
    2
)

# ==========================================================
# Claims
# ==========================================================

total_claims = len(claim)

total_claim_amount = round(
    claim["claim_amount"].sum(),
    2
)

avg_claim = round(
    claim["claim_amount"].mean(),
    2
)

fraud_cases = len(
    claim[
        claim["fraud_risk"]=="High"
    ]
)

# ==========================================================
# Underwriting
# ==========================================================

auto_approved = len(

    uw[
        uw["underwriting_decision"]=="Approved"
    ]

)

manual_review = len(

    uw[
        uw["manual_review_flag"]==True
    ]

)

avg_review_time = round(

    uw["review_time_minutes"].mean(),

    2

)

avg_risk = round(

    uw["risk_score"].mean(),

    2

)

# ==========================================================
# AI
# ==========================================================

ai_sessions = len(ai)

avg_ai_confidence = round(

    ai["confidence_score"].mean()*100,

    2

)

avg_rating = round(

    ai["customer_rating"].mean(),

    2

)

escalation = len(

    ai[
        ai["escalation_required"]==True
    ]

)

# ==========================================================
# Journey
# ==========================================================

drop_off = len(

    journey[
        journey["abandoned_flag"]==True
    ]

)

avg_duration = round(

    journey["duration_seconds"].mean(),

    2

)

# ==========================================================
# Executive Dashboard
# ==========================================================

dashboard = pd.DataFrame({

"Metric":[

"Total Quotes",
"Policies Issued",
"Conversion Rate",

"Average Premium",

"Total Claims",

"Total Claim Amount",

"Average Claim Amount",

"Fraud Cases",

"Auto Approved",

"Manual Review",

"Average Review Time",

"Average Risk Score",

"AI Sessions",

"Average AI Confidence",

"Average Customer Rating",

"Escalations",

"Customer Drop-offs",

"Average Journey Time"

],

"Value":[

total_quotes,

converted,

conversion_rate,

avg_premium,

total_claims,

total_claim_amount,

avg_claim,

fraud_cases,

auto_approved,

manual_review,

avg_review_time,

avg_risk,

ai_sessions,

avg_ai_confidence,

avg_rating,

escalation,

drop_off,

avg_duration

]

})

# ==========================================================
# Save
# ==========================================================

dashboard.to_csv(

    MART/"executive_dashboard.csv",

    index=False

)

dashboard.to_parquet(

    MART/"executive_dashboard.parquet",

    index=False

)

print("="*60)

print("Executive Dashboard Mart Created")

print("="*60)

print(dashboard)

Executive Dashboard Mart Created
                     Metric         Value
0              Total Quotes  3.811090e+05
1           Policies Issued  4.671000e+04
2           Conversion Rate  1.226000e+01
3           Average Premium  4.841130e+03
4              Total Claims  9.765500e+04
5        Total Claim Amount  2.872511e+08
6      Average Claim Amount  2.941490e+03
7               Fraud Cases  1.264000e+03
8             Auto Approved  3.087200e+04
9             Manual Review  3.586000e+04
10      Average Review Time  2.596000e+01
11       Average Risk Score  5.202000e+01
12              AI Sessions  7.629690e+05
13    Average AI Confidence  8.950000e+01
14  Average Customer Rating  4.350000e+00
15              Escalations  7.658300e+04
16       Customer Drop-offs  3.343990e+05
17     Average Journey Time  1.270300e+02


underwriting_dashboard

In [2]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

uw = pd.read_csv(FACT / "fact_underwriting.csv")
policy = pd.read_csv(DIM / "dim_policy.csv")
vehicle = pd.read_csv(DIM / "dim_vehicle.csv")

# ==========================================================
# Merge Dimensions
# ==========================================================

dashboard = (
    uw
    .merge(
        policy[
            [
                "policy_sk",
                "policy_type",
                "coverage_type",
                "policy_status"
            ]
        ],
        on="policy_sk",
        how="left"
    )
    .merge(
        vehicle[
            [
                "vehicle_sk",
                "make",
                "model",
                "fuel_type",
                "segment",
                "vehicle_age_band"
            ]
        ],
        on="vehicle_sk",
        how="left"
    )
)

# ==========================================================
# Dashboard KPIs
# ==========================================================

dashboard["auto_approved_flag"] = (
    dashboard["underwriting_decision"] == "Approved"
).astype(int)

dashboard["manual_review_flag"] = (
    dashboard["manual_review_flag"]
).astype(int)

dashboard["rejected_flag"] = (
    dashboard["underwriting_decision"] == "Rejected"
).astype(int)

dashboard["stp_flag"] = (
    dashboard["processing_mode"] == "Straight Through Processing"
).astype(int)

dashboard["sla_breached_flag"] = (
    dashboard["sla_status"] == "SLA Breached"
).astype(int)

dashboard["high_risk_flag"] = (
    dashboard["risk_band"].isin(
        ["High", "Very High"]
    )
).astype(int)

dashboard["fraud_flag"] = (
    dashboard["fraud_probability"] >= 0.70
).astype(int)

# ==========================================================
# Final Dashboard Table
# ==========================================================

underwriting_dashboard = dashboard[
    [
        "underwriting_sk",
        "policy_sk",
        "customer_sk",
        "vehicle_sk",

        "policy_type",
        "coverage_type",
        "policy_status",

        "make",
        "model",
        "segment",
        "fuel_type",
        "vehicle_age_band",

        "risk_score",
        "risk_band",

        "fraud_probability",

        "underwriting_decision",

        "manual_review_flag",
        "auto_approved_flag",
        "rejected_flag",
        "stp_flag",

        "review_time_minutes",

        "premium_adjustment_pct",

        "rules_triggered",

        "ai_confidence",

        "processing_mode",

        "underwriter",

        "sla_status",
        "sla_breached_flag",

        "high_risk_flag",
        "fraud_flag",

        "recommendation",

        "model_version",

        "underwriting_date_sk"
    ]
]

# ==========================================================
# Save
# ==========================================================

underwriting_dashboard.to_csv(
    MART / "underwriting_dashboard.csv",
    index=False
)

underwriting_dashboard.to_parquet(
    MART / "underwriting_dashboard.parquet",
    index=False
)

print("=" * 70)
print("Underwriting Dashboard Mart Created Successfully")
print("=" * 70)

print(underwriting_dashboard.head())

print(f"\nRows    : {len(underwriting_dashboard):,}")
print(f"Columns : {len(underwriting_dashboard.columns)}")

Underwriting Dashboard Mart Created Successfully
   underwriting_sk  policy_sk  customer_sk  vehicle_sk    policy_type  \
0                1          1            1           1  Comprehensive   
1                2          2            2           2  Comprehensive   
2                3          3            3           3  Comprehensive   
3                4          4            4           4  Comprehensive   
4                5          5            5           5    Third Party   

  coverage_type policy_status  make model segment  ... ai_confidence  \
0          Gold        Active     1    M1       A  ...         83.06   
1         Basic       Expired     1    M1       A  ...         82.54   
2          Gold        Active     1    M1       A  ...         83.03   
3         Basic        Active     1    M2      C1  ...         95.99   
4         Basic        Active     2    M3       A  ...         85.81   

               processing_mode  underwriter    sla_status  sla_breached_flag  \

customer_dashboard

In [3]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")

FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

MART.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

customer = pd.read_csv(DIM / "dim_customer.csv")

quote = pd.read_csv(FACT / "fact_quote.csv")

journey = pd.read_csv(FACT / "fact_customer_journey.csv")

ai = pd.read_csv(FACT / "fact_ai_interaction.csv")

underwriting = pd.read_csv(FACT / "fact_underwriting.csv")

# ==========================================================
# Quote Metrics
# ==========================================================

quote_summary = (
    quote
    .groupby("customer_sk")
    .agg(
        total_quotes=("quote_sk", "count"),
        avg_premium=("quoted_premium", "mean"),
        converted=("conversion_flag", "max")
    )
    .reset_index()
)

# ==========================================================
# Journey Metrics
# ==========================================================

journey_summary = (
    journey
    .groupby("customer_sk")
    .agg(
        total_journey_events=("journey_sk", "count"),
        avg_session_duration=("duration_seconds", "mean"),
        abandoned_sessions=("abandoned_flag", "sum"),
        ai_assistance_used=("ai_assistance_used", "max")
    )
    .reset_index()
)

# ==========================================================
# AI Interaction Metrics
# ==========================================================

ai_summary = (
    ai
    .groupby("customer_sk")
    .agg(
        total_ai_interactions=("interaction_sk", "count"),
        avg_ai_confidence=("confidence_score", "mean"),
        avg_customer_rating=("customer_rating", "mean"),
        escalations=("escalation_required", "sum"),
        avg_response_time=("response_time_ms", "mean")
    )
    .reset_index()
)

# ==========================================================
# Underwriting Metrics
# ==========================================================

uw_summary = (
    underwriting
    .groupby("customer_sk")
    .agg(
        avg_risk_score=("risk_score", "mean"),
        manual_reviews=("manual_review_flag", "sum"),
        avg_review_time=("review_time_minutes", "mean"),
        fraud_probability=("fraud_probability", "max")
    )
    .reset_index()
)

# ==========================================================
# Merge Everything
# ==========================================================

customer_dashboard = (
    customer
    .merge(
        quote_summary,
        on="customer_sk",
        how="left"
    )
    .merge(
        journey_summary,
        on="customer_sk",
        how="left"
    )
    .merge(
        ai_summary,
        on="customer_sk",
        how="left"
    )
    .merge(
        uw_summary,
        on="customer_sk",
        how="left"
    )
)

# ==========================================================
# Fill Missing Values
# ==========================================================

customer_dashboard.fillna(
    {
        "total_quotes":0,
        "avg_premium":0,
        "converted":0,
        "total_journey_events":0,
        "avg_session_duration":0,
        "abandoned_sessions":0,
        "ai_assistance_used":False,
        "total_ai_interactions":0,
        "avg_ai_confidence":0,
        "avg_customer_rating":0,
        "escalations":0,
        "avg_response_time":0,
        "avg_risk_score":0,
        "manual_reviews":0,
        "avg_review_time":0,
        "fraud_probability":0
    },
    inplace=True
)

# ==========================================================
# Derived KPIs
# ==========================================================

customer_dashboard["customer_segment"] = pd.cut(

    customer_dashboard["avg_premium"],

    bins=[0,5000,10000,20000,float("inf")],

    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
        "Premium"
    ]

)

customer_dashboard["conversion_status"] = customer_dashboard[
    "converted"
].map({
    1:"Converted",
    0:"Not Converted"
})

customer_dashboard["engagement_level"] = pd.cut(

    customer_dashboard["total_ai_interactions"],

    bins=[-1,0,2,5,100],

    labels=[
        "No Engagement",
        "Low",
        "Medium",
        "High"
    ]

)

customer_dashboard["risk_category"] = pd.cut(

    customer_dashboard["avg_risk_score"],

    bins=[0,30,60,80,100],

    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]

)

# ==========================================================
# Save
# ==========================================================

customer_dashboard.to_csv(
    MART / "customer_dashboard.csv",
    index=False
)

customer_dashboard.to_parquet(
    MART / "customer_dashboard.parquet",
    index=False
)

print("=" * 70)
print("Customer Dashboard Mart Created Successfully")
print("=" * 70)

print(customer_dashboard.head())

print(f"\nRows    : {len(customer_dashboard):,}")
print(f"Columns : {len(customer_dashboard.columns)}")

Customer Dashboard Mart Created Successfully
   customer_sk customer_id  gender  customer_age age_band  \
0            1  CUST000001    Male            44    36-45   
1            2  CUST000002    Male            76      60+   
2            3  CUST000003    Male            47    46-60   
3            4  CUST000004    Male            21    18-25   
4            5  CUST000005  Female            29    26-35   

   has_driving_license driving_license_status  previously_insured  \
0                    1               Licensed                   0   
1                    1               Licensed                   0   
2                    1               Licensed                   0   
3                    1               Licensed                   1   
4                    1               Licensed                   1   

    insurance_history customer_segment  ...  avg_customer_rating escalations  \
0    First Time Buyer     Medium Value  ...             4.333333           1   
1    First Ti

Sales Dashboard

In [5]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")

FACT_PATH = GOLD_PATH / "facts"
DIM_PATH = GOLD_PATH / "dimensions"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Data
# ==========================================================

quote = pd.read_csv(SILVER_PATH / "quote.csv")

channel = pd.read_csv(DIM_PATH / "dim_channel.csv")

print(f"Quote Records   : {len(quote):,}")
print(f"Channel Records : {len(channel):,}")

# ==========================================================
# Data Type Alignment
# ==========================================================

quote["sales_channel"] = quote["sales_channel"].astype(int)

channel["channel_id"] = channel["channel_id"].astype(int)

quote["quote_date"] = pd.to_datetime(quote["quote_date"])

# ==========================================================
# Date Key
# ==========================================================

quote["date_sk"] = (
    quote["quote_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# ==========================================================
# Channel Lookup
# ==========================================================

quote = quote.merge(

    channel[
        [
            "channel_sk",
            "channel_id"
        ]
    ],

    left_on="sales_channel",

    right_on="channel_id",

    how="left"

)

# ==========================================================
# Business Logic
# ==========================================================

quote["conversion_flag"] = quote["accepted_offer"].astype(int)

quote["quote_value_band"] = pd.cut(

    quote["quoted_premium"],

    bins=[0,5000,10000,20000,float("inf")],

    labels=[

        "Low",

        "Medium",

        "High",

        "Premium"

    ]

)

# ==========================================================
# Fact Table
# ==========================================================

fact_quote = quote[

    [

        "quote_sk",

        "quote_id",

        "customer_sk",

        "channel_sk",

        "date_sk",

        "quoted_premium",

        "accepted_offer",

        "conversion_flag",

        "quote_status",

        "quote_stage",

        "device_type",

        "quote_source",

        "days_since_last_contact",

        "quote_value_band"

    ]

]

# ==========================================================
# Save
# ==========================================================

fact_quote.to_csv(

    FACT_PATH / "fact_quote.csv",

    index=False

)

fact_quote.to_parquet(

    FACT_PATH / "fact_quote.parquet",

    index=False

)

print("="*60)

print("FACT_QUOTE CREATED SUCCESSFULLY")

print("="*60)

print(fact_quote.head())

print(f"\nRows    : {len(fact_quote):,}")

print(f"Columns : {len(fact_quote.columns)}")

ImportError: numpy._core.multiarray failed to import

Claims Dashboard

KeyError: 'customer_sk'